In [76]:
# set up environment
import sys
import json
import re
import requests
import pandas as pd
from pathlib import Path
import json
import csv
from caveclient import CAVEclient
import urllib.parse
#pip install neuprint-python
from neuprint import Client, fetch_neurons, fetch_custom, NeuronCriteria as NC


print("Python executable:", sys.executable)
print("Imports OK")

Python executable: c:\Users\jsfdz\Documents\Python\cave_env\Scripts\python.exe
Imports OK


## set up paths and directories

In [77]:
# set up directories
PROJECT_ROOT = Path.cwd() #anchor to current notebook location

TABLES_DIR = PROJECT_ROOT /"analysis_MANC v1.2.3_04B_cross_neuromeres"/"neuromere_tables"

OUTPUT_DIR = PROJECT_ROOT /"analysis_MANC v1.2.3_MN_connectivity" / "MN_analysis_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Tables dir :", TABLES_DIR)
print("Output dir :", OUTPUT_DIR)

#SECOND_OUTPUT_DIR = PROJECT_ROOT /"analysis_MANC v1.2.3_04B_MN_connectivity"/"MN_tables"
#SECOND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Tables dir : c:\Users\jsfdz\4B-analysis\4B-analysis\analysis_MANC v1.2.3_04B_cross_neuromeres\neuromere_tables
Output dir : c:\Users\jsfdz\4B-analysis\4B-analysis\analysis_MANC v1.2.3_MN_connectivity\MN_analysis_outputs


In [78]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from current working directory

token = os.environ["NEUPRINT_APPLICATION_CREDENTIALS"]

In [79]:
from neuprint import Client

c = Client(
    "https://neuprint.janelia.org",
    dataset="manc:v1.2.3",
    token=token
)
print(c)

Client("https://neuprint.janelia.org", "manc:v1.2.3")


## Inspect CSV

In [ ]:
# Retreive CSV files, load

# csv_path = (
#     PROJECT_ROOT
#     / "analysis_MANC v1.2.3_04B_cross_neuromeres"
#     / "meta_analysis_outputs"
#     / "pooled_all_neuromeres_classification_04B.csv"
# )

#let's first do only one neuromere to make sure it is working well
csv_path = (
    PROJECT_ROOT
    / "analysis_MANC v1.2.3_04B_cross_neuromeres"
    / "neuromere_tables"
    / "Classification_04B_T1L.csv"
)


print("CSV file:")
print(" -", csv_path.name)

assert csv_path.exists(), f"File not found: {csv_path}"

CSV file:
 - Classification_04B_T1L.csv


In [140]:
#find mn connectivity 
#  from CSV
df = pd.read_csv(csv_path)

# Extract body IDs 
my_ids = df["bodyId"].dropna().astype(int).unique().tolist()
print(f"Loaded {len(my_ids)} neurons from {csv_path.name}")

# Convert list to Cypher format 
my_ids_cypher = "[" + ",".join(map(str, my_ids[:5000])) + "]"
# ↑ limit size if needed (neuPrint can choke on huge lists); not necessary for this single neuromere but it won't harm


# Cypher query 

query = f"""
WITH {my_ids_cypher} AS neuronList
UNWIND neuronList AS bodyId

MATCH (n:Neuron {{bodyId: bodyId}})-[c:ConnectsTo]->(down:Neuron)

WHERE (down.type CONTAINS "motor" OR down.class = "motor")
  AND c.weight >= 1   //

RETURN 
    bodyId AS upstream,
    down.bodyId AS motorNeuron,
    down.type AS motorType,
    c.weight AS synapseCount
ORDER BY upstream, synapseCount DESC
"""

c
df_results = c.fetch_custom(query)

print(df_results.head())


Loaded 97 neurons from Classification_04B_T1L.csv
   upstream  motorNeuron                         motorType  synapseCount
0     10024        12686  Tergopleural/Pleural promotor MN             1
1     10314        12686  Tergopleural/Pleural promotor MN             3
2     10314        13490  Tergopleural/Pleural promotor MN             1
3     10314        12396  Tergopleural/Pleural promotor MN             1
4     11143        12686  Tergopleural/Pleural promotor MN             1


In [151]:
# Inspect to learn  more about hte Mns that 04B connects to:
mn_ids = df_results['motorNeuron'].unique().tolist()

#neuprint query to retrevie MN metadata

mn_ids_cypher = str(mn_ids)  # converts list → Cypher format

query_mn = f"""
WITH {mn_ids_cypher} AS mnList
UNWIND mnList AS bodyId

MATCH (n:Neuron {{bodyId: bodyId}}) 

RETURN
    n.bodyId AS bodyId,
    n.type AS type,
    n.instance AS instance,
    n.class AS class,
    n.subclass AS subclass,
    n.hemilineage AS hemilineage,
    n.somaNeuromere AS somaNeuromere,
    n.somaSide AS somaSide,
    n.rootSide AS rootSide,
    n.longTract AS longTract,
    n.birthtime AS birthtime
ORDER BY type
"""
df_mn_info = c.fetch_custom(query_mn)
df_mn_info.head()



#how much input does each mn receive from the individual 04b neurons
mn_strength = df_results.groupby('motorNeuron')['synapseCount'].sum().reset_index()

df_mn_info = df_mn_info.merge(
    mn_strength,
    left_on='bodyId',
    right_on='motorNeuron',
    how='left'
)

#print (df_mn_info.head())
#df_results['motorNeuron'].nunique()
#mn_ids = df_results['motorNeuron'].unique()
#len(mn_ids)

#for mn in mn_ids:
   # print(mn)

print (df_mn_info)

#how many -4B neurons talk to each mn 
# mn_input_counts = df_results.groupby('motorNeuron')['upstream'].nunique()
# #map to mn table 
# df_mn_info['num_04B_neurons'] = df_mn_info['bodyId'].map(mn_input_counts)
# #view
# df_mn_info[['bodyId', 'type', 'num_04B_neurons']]
print("MNs from connectivity:", len(df_results['motorNeuron'].unique()))
print("MNs in df_mn_info:", len(df_mn_info))

df_mn_info['num_04B_neurons'] = df_mn_info['bodyId'].map(
    df_results.groupby('motorNeuron')['upstream'].nunique()
)

df_mn_info

   bodyId                              type        instance         class  \
0   12096       Pleural remotor/abductor MN  MNfl31_ProAN_L  motor neuron   
1   13628       Pleural remotor/abductor MN  MNfl17_ProAN_L  motor neuron   
2   12686  Tergopleural/Pleural promotor MN  MNfl54_DProN_L  motor neuron   
3   13490  Tergopleural/Pleural promotor MN  MNfl53_DProN_L  motor neuron   
4   12396  Tergopleural/Pleural promotor MN  MNfl56_DProN_L  motor neuron   
5   19319  Tergopleural/Pleural promotor MN  MNfl55_DProN_L  motor neuron   
6   13464  Tergopleural/Pleural promotor MN  MNfl53_DProN_R  motor neuron   
7   16250  Tergopleural/Pleural promotor MN  MNfl54_DProN_R  motor neuron   
8   12628  Tergopleural/Pleural promotor MN  MNfl55_DProN_R  motor neuron   

  subclass hemilineage somaNeuromere somaSide rootSide longTract  \
0       fl         TBD            T1      LHS     None      None   
1       fl         TBD            T1      LHS     None      None   
2       fl         TBD   

,bodyId,type,instance,class,subclass,hemilineage,somaNeuromere,somaSide,rootSide,longTract,birthtime,motorNeuron,synapseCount,num_04B_neurons
0,12096,Pleural remotor/abductor MN,MNfl31_ProAN_L,motor neuron,fl,TBD,T1,LHS,None,None,early secondary,12096,76,26
1,13628,Pleural remotor/abductor MN,MNfl17_ProAN_L,motor neuron,fl,TBD,T1,LHS,None,None,early secondary,13628,82,19
2,12686,Tergopleural/Pleural promotor MN,MNfl54_DProN_L,motor neuron,fl,TBD,T1,LHS,None,None,early secondary,12686,1473,63
3,13490,Tergopleural/Pleural promotor MN,MNfl53_DProN_L,motor neuron,fl,TBD,T1,LHS,None,None,secondary,13490,1067,62
4,12396,Tergopleural/Pleural promotor MN,MNfl56_DProN_L,motor neuron,fl,08A,T1,LHS,None,None,early secondary,12396,377,48
5,19319,Tergopleural/Pleural promotor MN,MNfl55_DProN_L,motor neuron,fl,TBD,T1,LHS,None,None,early secondary,19319,684,56
6,13464,Tergopleural/Pleural promotor MN,MNfl53_DProN_R,motor neuron,fl,TBD,T1,RHS,None,None,secondary,13464,10,3
7,16250,Tergopleural/Pleural promotor MN,MNfl54_DProN_R,motor neuron,fl,TBD,T1,RHS,None,None,early secondary,16250,4,1
8,12628,Tergopleural/Pleural promotor MN,MNfl55_DProN_R,motor neuron,fl,TBD,T1,RHS,None,None,early secondary,12628,1,1


In [153]:
df_results['morphology_cluster_id'] = df_results['upstream'].map(
    df.set_index('bodyId')['morphology_cluster_id']
)

mn_clusters = df_results.groupby('motorNeuron')['morphology_cluster_id'] \
    .unique() \
    .reset_index(name='morph_clusters')

df_mn_info = df_mn_info.merge(
    mn_clusters,
    left_on='bodyId',
    right_on='motorNeuron',
    how='left'
)

df_mn_info[['bodyId', 'type', 'morph_clusters']]

,bodyId,type,morph_clusters
0,12096,Pleural remotor/abductor MN,"[IA_1_1, IR_1_5, IR_2_6, IR_2_1, BR_1_1, IR_2_..."
1,13628,Pleural remotor/abductor MN,"[IR_1_5, BR_1_1, BI_1_1, IR_2_2, IR_2_3, IR_2_4]"
2,12686,Tergopleural/Pleural promotor MN,"[BR_1_1, IA_1_1, BI_1_1, IR_1_5, IR_1_2, IR_2_..."
3,13490,Tergopleural/Pleural promotor MN,"[IA_1_1, BR_1_1, IR_1_5, IR_1_2, BI_1_1, IR_2_..."
4,12396,Tergopleural/Pleural promotor MN,"[IA_1_1, IR_1_5, BR_1_1, IR_1_2, BI_1_1, IR_2_..."
5,19319,Tergopleural/Pleural promotor MN,"[IR_1_5, BR_1_1, IR_1_2, BI_1_1, IR_2_6, IR_2_..."
6,13464,Tergopleural/Pleural promotor MN,"[BI_1_1, BR_1_1]"
7,16250,Tergopleural/Pleural promotor MN,[BR_1_1]
8,12628,Tergopleural/Pleural promotor MN,[BR_1_1]


## Review query

In [82]:
print(df_results.columns)

Index(['upstream', 'motorNeuron', 'motorType', 'synapseCount'], dtype='object')


In [83]:
print(df_results.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279 entries, 0 to 278
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   upstream      279 non-null    int64 
 1   motorNeuron   279 non-null    int64 
 2   motorType     279 non-null    object
 3   synapseCount  279 non-null    int64 
dtypes: int64(3), object(1)
memory usage: 8.8+ KB
None


In [84]:
id_map = df[["bodyId", "major_class"]].drop_duplicates()

df_merged = df_results.merge(
    id_map,
    left_on="upstream",
    right_on="bodyId",
    how="left"
)

In [85]:
conn_summary = (
    df_merged
    .groupby(["major_class", "motorType"])
    .agg(total_synapses=("synapseCount", "sum"),
         n_connections=("motorNeuron", "count"))
    .reset_index()
)

print(conn_summary)

  major_class                         motorType  total_synapses  n_connections
0          BI       Pleural remotor/abductor MN               2              2
1          BI  Tergopleural/Pleural promotor MN             149             17
2          BR       Pleural remotor/abductor MN              77             14
3          BR  Tergopleural/Pleural promotor MN            1296             74
4          IA       Pleural remotor/abductor MN               1              1
5          IA  Tergopleural/Pleural promotor MN              14              6
6          IR       Pleural remotor/abductor MN              78             28
7          IR  Tergopleural/Pleural promotor MN            2157            137


In [86]:
n_connected = df_results["upstream"].nunique()
print(n_connected)

83


In [70]:
connected_ids = set(df_results["upstream"])
all_ids = set(my_ids)

not_connected = all_ids - connected_ids

print(f"{len(not_connected)} neurons have NO motor connections")

df_connected = df[df["bodyId"].isin(connected_ids)]

print(df_connected["major_class"].value_counts())

42 neurons have NO motor connections
major_class
IR    34
BR    18
BI     2
IA     1
Name: count, dtype: int64


In [71]:
# total per class
total_per_class = df["major_class"].value_counts()

# connected per class
connected_per_class = df_connected["major_class"].value_counts()

fraction_connected = (connected_per_class / total_per_class).fillna(0)

print(fraction_connected.sort_values(ascending=False))

major_class
BR    0.692308
IR    0.566667
BI    0.333333
IA    0.250000
BA    0.000000
Name: count, dtype: float64


In [75]:
results_counts = []
results_frac = []

total_per_class = df["major_class"].value_counts()

for t in thresholds:
    df_t = df_results[df_results["synapseCount"] >= t]
    ids_t = set(df_t["upstream"])
    df_meta_t = df[df["bodyId"].isin(ids_t)]
    
    # --- raw counts (FIXED) ---
    counts = df_meta_t["major_class"].value_counts()
    counts = counts.reindex(total_per_class.index, fill_value=0)
    counts.name = f">={t}"
    results_counts.append(counts)
    
    # --- fractions ---
    frac = (counts / total_per_class).fillna(0)
    frac.name = f">={t}"
    results_frac.append(frac)

counts_table = pd.concat(results_counts, axis=1)
frac_table = pd.concat(results_frac, axis=1)

print("=== Raw counts ===")
print(counts_table)

print("\n=== Fractions ===")
print(frac_table)

=== Raw counts ===
             >=1  >=10  >=20
major_class                 
IR            49    24    18
BR            25    13    10
BI             6     2     2
IA             3     0     0
BA             0     0     0

=== Fractions ===
                  >=1      >=10      >=20
major_class                              
IR           0.816667  0.400000  0.300000
BR           0.961538  0.500000  0.384615
BI           1.000000  0.333333  0.333333
IA           0.750000  0.000000  0.000000
BA           0.000000  0.000000  0.000000


In [87]:
print("\n=== Connectivity summary ===")

for t in thresholds:
    df_t = df_results[df_results["synapseCount"] >= t]
    
    connected_ids = set(df_t["upstream"])
    n_connected = len(connected_ids)
    n_not = len(my_ids) - n_connected
    
    print(f"\nThreshold ≥{t}")
    print(f"  Connected:     {n_connected} / {len(my_ids)} ({n_connected/len(my_ids):.2%})")
    print(f"  Not connected: {n_not}")


=== Connectivity summary ===

Threshold ≥1
  Connected:     83 / 97 (85.57%)
  Not connected: 14

Threshold ≥10
  Connected:     39 / 97 (40.21%)
  Not connected: 58

Threshold ≥20
  Connected:     30 / 97 (30.93%)
  Not connected: 67


In [88]:
print("\n=== Connectivity by threshold and class ===")

for t in thresholds:
    df_t = df_results[df_results["synapseCount"] >= t]
    ids_t = set(df_t["upstream"])
    
    df_meta_t = df[df["bodyId"].isin(ids_t)]
    
    n_connected = len(ids_t)
    
    print(f"\n--- Threshold ≥{t} ---")
    print(f"Total: {n_connected} / {len(my_ids)} neurons")
    
    # counts per class
    counts = df_meta_t["major_class"].value_counts()
    
    # fractions within this threshold
    frac = df_meta_t["major_class"].value_counts(normalize=True)
    
    summary = pd.concat([counts, frac], axis=1)
    summary.columns = ["count", "fraction_within_threshold"]
    
    print(summary)


=== Connectivity by threshold and class ===

--- Threshold ≥1 ---
Total: 83 / 97 neurons
             count  fraction_within_threshold
major_class                                  
IR              49                   0.590361
BR              25                   0.301205
BI               6                   0.072289
IA               3                   0.036145

--- Threshold ≥10 ---
Total: 39 / 97 neurons
             count  fraction_within_threshold
major_class                                  
IR              24                   0.615385
BR              13                   0.333333
BI               2                   0.051282

--- Threshold ≥20 ---
Total: 30 / 97 neurons
             count  fraction_within_threshold
major_class                                  
IR              18                   0.600000
BR              10                   0.333333
BI               2                   0.066667


In [90]:
print("\n=== Average total synapses per neuron by class ===")

# total synapses per neuron
df_sum = (
    df_results
    .groupby("upstream")["synapseCount"]
    .sum()
    .reset_index()
)

# attach class info
df_sum = df_sum.merge(
    df[["bodyId", "major_class"]],
    left_on="upstream",
    right_on="bodyId",
    how="left"
)

all_classes = df["major_class"].unique()

for t in thresholds:
    ids_t = set(
        df_results[df_results["synapseCount"] >= t]["upstream"]
    )
    
    df_t = df_sum[df_sum["upstream"].isin(ids_t)]
    
    avg_by_class = (
        df_t
        .groupby("major_class")["synapseCount"]
        .mean()
        .reindex(all_classes)   # 👈 force all classes
    )
    
    print(f"\n--- Threshold ≥{t} ---")
    print(avg_by_class)


=== Average total synapses per neuron by class ===

--- Threshold ≥1 ---
major_class
IR    45.612245
BR    54.920000
BI    25.166667
IA     5.000000
BA          NaN
Name: synapseCount, dtype: float64

--- Threshold ≥10 ---
major_class
IR    85.333333
BR    98.000000
BI    68.500000
IA          NaN
BA          NaN
Name: synapseCount, dtype: float64

--- Threshold ≥20 ---
major_class
IR    104.277778
BR    121.200000
BI     68.500000
IA           NaN
BA           NaN
Name: synapseCount, dtype: float64


# try to analyze at one step deeper in the morphological clusters

In [92]:
class_pairs = (
    df[["major_class", "subclass_level1"]]
    .drop_duplicates()
    .sort_values(["major_class", "subclass_level1"])
)

full_index = pd.MultiIndex.from_frame(class_pairs)

In [93]:
results_counts = []
results_frac = []

total_per_group = (
    df
    .groupby(["major_class", "subclass_level1"])
    .size()
)

for t in thresholds:
    df_t = df_results[df_results["synapseCount"] >= t]
    ids_t = set(df_t["upstream"])
    
    df_meta_t = df[df["bodyId"].isin(ids_t)]
    
    counts = (
        df_meta_t
        .groupby(["major_class", "subclass_level1"])
        .size()
        .reindex(full_index, fill_value=0)
    )
    counts.name = f">={t}"
    results_counts.append(counts)
    
    frac = (counts / total_per_group).fillna(0)
    frac.name = f">={t}"
    results_frac.append(frac)

counts_table = pd.concat(results_counts, axis=1)
frac_table = pd.concat(results_frac, axis=1)

print(counts_table)
print(frac_table)

                             >=1  >=10  >=20
major_class subclass_level1                 
BA          1                  0     0     0
BI          1                  6     2     2
BR          1                 25    13    10
IA          1                  3     0     0
IR          1                 20     9     7
            2                 29    15    11
                                  >=1      >=10      >=20
major_class subclass_level1                              
BA          1                0.000000  0.000000  0.000000
BI          1                1.000000  0.333333  0.333333
BR          1                0.961538  0.500000  0.384615
IA          1                0.750000  0.000000  0.000000
IR          1                0.666667  0.300000  0.233333
            2                0.966667  0.500000  0.366667


In [94]:
df_sum = (
    df_results
    .groupby("upstream")["synapseCount"]
    .sum()
    .reset_index()
)

df_sum = df_sum.merge(
    df[["bodyId", "major_class", "subclass_level1"]],
    left_on="upstream",
    right_on="bodyId",
    how="left"
)

for t in thresholds:
    ids_t = set(
        df_results[df_results["synapseCount"] >= t]["upstream"]
    )
    
    df_t = df_sum[df_sum["upstream"].isin(ids_t)]
    
    avg_strength = (
        df_t
        .groupby(["major_class", "subclass_level1"])["synapseCount"]
        .mean()
        .reindex(full_index)
    )
    
    print(f"\n=== Avg strength ≥{t} ===")
    print(avg_strength)


=== Avg strength ≥1 ===
major_class  subclass_level1
BA           1                        NaN
BI           1                  25.166667
BR           1                  54.920000
IA           1                   5.000000
IR           1                  49.400000
             2                  43.000000
Name: synapseCount, dtype: float64

=== Avg strength ≥10 ===
major_class  subclass_level1
BA           1                         NaN
BI           1                   68.500000
BR           1                   98.000000
IA           1                         NaN
IR           1                  100.555556
             2                   76.200000
Name: synapseCount, dtype: float64

=== Avg strength ≥20 ===
major_class  subclass_level1
BA           1                         NaN
BI           1                   68.500000
BR           1                  121.200000
IA           1                         NaN
IR           1                  119.285714
             2                   94.72727

# now use the refined morphological clsuter ID

In [95]:
# ensure clean IDs (important if NaNs exist)
df_clean = df.dropna(subset=["morphology_cluster_id"])

all_clusters = sorted(df_clean["morphology_cluster_id"].unique())

In [96]:
results_counts = []
results_frac = []

# total neurons per cluster
total_per_cluster = df_clean["morphology_cluster_id"].value_counts()

for t in thresholds:
    df_t = df_results[df_results["synapseCount"] >= t]
    ids_t = set(df_t["upstream"])
    
    df_meta_t = df_clean[df_clean["bodyId"].isin(ids_t)]
    
    # counts
    counts = df_meta_t["morphology_cluster_id"].value_counts()
    counts = counts.reindex(all_clusters, fill_value=0)
    counts.name = f">={t}"
    results_counts.append(counts)
    
    # fractions
    frac = (counts / total_per_cluster).fillna(0)
    frac.name = f">={t}"
    results_frac.append(frac)

counts_table = pd.concat(results_counts, axis=1)
frac_table = pd.concat(results_frac, axis=1)

print("=== Counts by morphology cluster ===")
print(counts_table)

print("\n=== Fraction per cluster ===")
print(frac_table)

=== Counts by morphology cluster ===
                       >=1  >=10  >=20
morphology_cluster_id                 
BA_1_1                   0     0     0
BI_1_1                   6     2     2
BR_1_1                  25    13    10
IA_1_1                   3     0     0
IR_1_1                   0     0     0
IR_1_2                   3     0     0
IR_1_3                   4     0     0
IR_1_4                   2     2     2
IR_1_5                  11     7     5
IR_2_1                   5     5     4
IR_2_2                  14     8     5
IR_2_3                   2     0     0
IR_2_4                   5     0     0
IR_2_5                   2     1     1
IR_2_6                   1     1     1

=== Fraction per cluster ===
                            >=1      >=10      >=20
morphology_cluster_id                              
BA_1_1                 0.000000  0.000000  0.000000
BI_1_1                 1.000000  0.333333  0.333333
BR_1_1                 0.961538  0.500000  0.384615
IA_1_1    

In [97]:
# total synapses per neuron
df_sum = (
    df_results
    .groupby("upstream")["synapseCount"]
    .sum()
    .reset_index()
)

df_sum = df_sum.merge(
    df_clean[["bodyId", "morphology_cluster_id"]],
    left_on="upstream",
    right_on="bodyId",
    how="left"
)

for t in thresholds:
    ids_t = set(
        df_results[df_results["synapseCount"] >= t]["upstream"]
    )
    
    df_t = df_sum[df_sum["upstream"].isin(ids_t)]
    
    avg_strength = (
        df_t
        .groupby("morphology_cluster_id")["synapseCount"]
        .mean()
        .reindex(all_clusters)
    )
    
    print(f"\n=== Avg strength ≥{t} ===")
    print(avg_strength.sort_values(ascending=False))


=== Avg strength ≥1 ===
morphology_cluster_id
IR_1_4    95.000000
IR_2_1    94.600000
IR_2_6    72.000000
IR_1_5    68.727273
BR_1_1    54.920000
IR_2_2    43.428571
IR_2_5    26.000000
BI_1_1    25.166667
IR_1_2     8.666667
IR_2_3     7.500000
IR_2_4     5.400000
IA_1_1     5.000000
IR_1_3     4.000000
BA_1_1          NaN
IR_1_1          NaN
Name: synapseCount, dtype: float64

=== Avg strength ≥10 ===
morphology_cluster_id
IR_1_5    102.142857
BR_1_1     98.000000
IR_1_4     95.000000
IR_2_1     94.600000
IR_2_6     72.000000
IR_2_2     69.125000
BI_1_1     68.500000
IR_2_5     45.000000
BA_1_1           NaN
IA_1_1           NaN
IR_1_1           NaN
IR_1_2           NaN
IR_1_3           NaN
IR_2_3           NaN
IR_2_4           NaN
Name: synapseCount, dtype: float64

=== Avg strength ≥20 ===
morphology_cluster_id
IR_1_5    129.00
BR_1_1    121.20
IR_2_1    109.75
IR_2_2     97.20
IR_1_4     95.00
IR_2_6     72.00
BI_1_1     68.50
IR_2_5     45.00
BA_1_1       NaN
IA_1_1       NaN
IR

# a heatmap to see if mf clsuters predict conenctivity strength or preferentially connect to mn types 